# Expectation Decider — Probability & Statistics Analysis

**Objective:** Analyze a synthetic dataset of 200 students and explore probability patterns associated with passing a competitive mathematics exam.

> **Important:** This dataset is synthetic, created for this assignment. Findings describe this generated sample only; they do not prove that any factor causes exam success.

## 0. Setup and load the dataset

The CSV is included in the same project folder. Run the notebook from the project root so the relative path works.

In [1]:
!pip install numpy pandas matplotlib seaborn scikit-learn scipy

  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl.metadata (18 kB)
  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl.metadata (142 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-13.0.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Usin


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import hypergeom, chi2_contingency

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("student_data.csv")
print("Dataset shape:", df.shape)
display(df.head())
display(df.describe(include="all"))

## 1. Understanding the basics

**Probability** is a numerical measure of how likely an event is, from 0 (impossible) to 1 (certain).

**Key terminology**
- **Experiment:** a process with an uncertain result, such as selecting one student at random.
- **Outcome:** one possible result, such as selecting a student who passed.
- **Sample space:** all possible outcomes of the experiment.
- **Event:** a set of outcomes, such as “the selected student passed.”
- **Complement:** the event that an event does not occur.
- **Joint event:** two conditions happen together.
- **Conditional probability:** the probability of an event when another event is known to have occurred.
- **Mutually exclusive events:** cannot occur together in the same trial.
- **Independent events:** knowing one event occurred does not change the probability of the other.

For a finite dataset, empirical probability is usually calculated as `number of observed cases / total number of cases`.

In [ ]:
N = len(df)
pass_count = (df["final_exam_pass"] == "Pass").sum()
fail_count = (df["final_exam_pass"] == "Fail").sum()

events = {
    "A: student passes": (df["final_exam_pass"] == "Pass").sum(),
    "B: attendance > 80%": (df["attendance"] > 80).sum(),
    "C: study > 10 hours/week": (df["study_hours"] > 10).sum(),
    "D: participates in group discussion": (df["group_discussion"] == "Yes").sum(),
}
event_summary = pd.DataFrame([
    {"Event": name, "Count": count, "Empirical probability": count / N}
    for name, count in events.items()
])
display(event_summary)

### Three example events from this dataset
1. A randomly selected student passes the final exam.
2. A randomly selected student studies more than 10 hours per week.
3. A randomly selected student both attends more than 80% of classes and passes.

The notebook calculates these probabilities from the actual generated records.

In [ ]:
p_pass = (df["final_exam_pass"] == "Pass").mean()
p_study_over_10 = (df["study_hours"] > 10).mean()
p_high_attendance_and_pass = (
    ((df["attendance"] > 80) & (df["final_exam_pass"] == "Pass")).sum() / N
)
print(f"P(Pass) = {p_pass:.4f}")
print(f"P(Study > 10 hours) = {p_study_over_10:.4f}")
print(f"P(Attendance > 80% AND Pass) = {p_high_attendance_and_pass:.4f}")

## 2. Empirical probability and theoretical probability

**Empirical example:** the probability of passing, estimated from the observed sample.

\[
P(\text{Pass}) = \frac{\text{number of students who passed}}{200}
\]

**Theoretical example tied to the selection setup:** if each of the 200 student records is equally likely to be selected, the probability of selecting one particular named record is:

\[
P(\text{that record}) = \frac{1}{200} = 0.005
\]

This theoretical probability relies on uniform random selection, while the pass probability above is estimated from observed outcomes.

In [ ]:
print(f"Empirical P(Pass) = {pass_count}/{N} = {pass_count/N:.4f}")
print(f"Theoretical P(select one specified record) = 1/{N} = {1/N:.4f}")

## 3. Random variable and probability distribution

Let **X = number of students who pass** among 3 students selected uniformly at random **without replacement** from this finite dataset.

Because sampling is without replacement, X follows a **hypergeometric distribution** with:
- population size `N = 200`
- number of passing students `K = pass_count`
- sample size `n = 3`

\[
P(X=x)=\frac{\binom{K}{x}\binom{N-K}{3-x}}{\binom{N}{3}},\quad x=0,1,2,3
\]

For a hypergeometric random variable:

\[
E[X]=n\frac{K}{N}
\]

\[
\operatorname{Var}(X)=n\frac{K}{N}\left(1-\frac{K}{N}\right)\frac{N-n}{N-1}
\]

In [ ]:
K = int((df["final_exam_pass"] == "Pass").sum())
sample_size = 3
x_values = np.arange(0, sample_size + 1)
distribution = hypergeom(M=N, n=K, N=sample_size)
probabilities = distribution.pmf(x_values)

dist_table = pd.DataFrame({
    "X = number of passes": x_values,
    "P(X = x)": probabilities
})
display(dist_table)
mean_x = distribution.mean()
variance_x = distribution.var()
print(f"Mean E[X] = {mean_x:.4f}")
print(f"Variance Var(X) = {variance_x:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=dist_table, x="X = number of passes", y="P(X = x)")
plt.title("Distribution of Passes Among 3 Randomly Selected Students")
plt.xlabel("Number of students who pass (X)")
plt.ylabel("Probability")
plt.tight_layout()
plt.show()

## 4. Venn diagram: study hours and attendance

Define:
- A = students studying more than 10 hours per week.
- B = students attending more than 80% of classes.
- A ∩ B = students satisfying both conditions.

The counts below are computed directly from the dataset. The diagram uses a two-set layout and labels the mutually exclusive regions: A only, overlap, B only, and neither.

In [ ]:
A = df["study_hours"] > 10
B = df["attendance"] > 80

a_only = int((A & ~B).sum())
both = int((A & B).sum())
b_only = int((~A & B).sum())
neither = int((~A & ~B).sum())

print(f"A only (study > 10, attendance <= 80): {a_only}")
print(f"A intersection B (both conditions): {both}")
print(f"B only (study <= 10, attendance > 80): {b_only}")
print(f"Neither condition: {neither}")
print("Check total:", a_only + both + b_only + neither, "=", N)

from matplotlib.patches import Circle
fig, ax = plt.subplots(figsize=(8, 5))
left = Circle((0.42, 0.52), 0.27, alpha=0.25, color="royalblue")
right = Circle((0.62, 0.52), 0.27, alpha=0.25, color="cornflowerblue")
ax.add_patch(left)
ax.add_patch(right)
ax.text(0.30, 0.83, "A: Study > 10 hours", ha="center", fontsize=10)
ax.text(0.74, 0.83, "B: Attendance > 80%", ha="center", fontsize=10)
ax.text(0.30, 0.52, f"A only\n{a_only}", ha="center", va="center")
ax.text(0.52, 0.52, f"Both\n{both}", ha="center", va="center", weight="bold")
ax.text(0.72, 0.52, f"B only\n{b_only}", ha="center", va="center")
ax.text(0.5, 0.08, f"Neither = {neither}", ha="center", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Venn Diagram of Study Hours and Attendance")
plt.tight_layout()
plt.show()

## 5. Contingency table and probability calculations

Create a two-way table comparing group-discussion participation with the final exam result.

\[
P(A \cap B)=\frac{\text{count of both A and B}}{N}
\]

\[
P(\text{Pass})=\frac{\text{all Pass cases}}{N}
\]

\[
P(\text{Pass}\mid\text{Discussion=Yes})=
\frac{\text{Discussion=Yes and Pass}}{\text{Discussion=Yes}}
\]

In [ ]:
contingency = pd.crosstab(df["group_discussion"], df["final_exam_pass"])
# Ensure columns appear even if one category is absent in a sample
contingency = contingency.reindex(index=["Yes", "No"], columns=["Pass", "Fail"], fill_value=0)
display(contingency)

joint_count = int(contingency.loc["Yes", "Pass"])
yes_total = int(contingency.loc["Yes"].sum())
pass_total = int(contingency["Pass"].sum())

p_joint = joint_count / N
p_marginal_pass = pass_total / N
p_pass_given_discussion = joint_count / yes_total if yes_total else np.nan

print(f"P(Discussion=Yes AND Pass) = {joint_count}/{N} = {p_joint:.4f}")
print(f"P(Pass) = {pass_total}/{N} = {p_marginal_pass:.4f}")
print(f"P(Pass | Discussion=Yes) = {joint_count}/{yes_total} = {p_pass_given_discussion:.4f}")

In [ ]:
contingency.plot(kind="bar", stacked=True, figsize=(7, 4))
plt.title("Final Exam Outcome by Group Discussion Participation")
plt.xlabel("Participates in group discussion")
plt.ylabel("Number of students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Relationships: conditional probability, independence, and exclusivity

**Interpretation:** `P(Pass | Discussion=Yes)` means: restrict attention to students who participated in group discussions, then calculate the fraction of that group who passed. It is not the same as the overall pass rate.

- **Mutually exclusive?** No. Participation and passing can occur together; the joint count is directly shown in the contingency table.
- **Independent?** They are independent only if `P(Pass | Discussion=Yes)` equals `P(Pass)` (equivalently, the joint probability equals the product of the marginals). Compare the sample estimates below.
- A chi-square test is included as an additional check of association. A small p-value can be evidence of association in this sample, but it does not establish causation.

In [ ]:
independent_by_estimates = np.isclose(p_pass_given_discussion, p_marginal_pass, atol=1e-12)
print(f"P(Pass | Discussion=Yes) = {p_pass_given_discussion:.4f}")
print(f"P(Pass) = {p_marginal_pass:.4f}")
print("Sample estimates suggest:", "approximately independent" if independent_by_estimates else "dependent")

chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"Chi-square statistic = {chi2:.4f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p_value:.4f}")
print("Interpret the p-value alongside sample size and assumptions; association does not prove causation.")

## 7. Bayes' theorem application (assignment-provided assumptions)

Let:
- P = student passes
- H = student has high attendance (>80%)

Given by the assignment:
- `P(H | P) = 0.70`
- `P(H | not P) = 0.40`
- `P(H) = 0.60`

First infer the prior pass probability using the law of total probability:

\[
P(H)=P(H|P)P(P)+P(H|\neg P)P(\neg P)
\]

Let `p = P(P)`. Then:

\[
0.60=0.70p+0.40(1-p)=0.40+0.30p
\]
\[
p=\frac{0.20}{0.30}=\frac{2}{3}
\]

Now apply Bayes' theorem:

\[
P(P|H)=\frac{P(H|P)P(P)}{P(H)}
=\frac{0.70\times(2/3)}{0.60}
=\frac{7}{9}\approx0.7778
\]

**Result:** Under the assignment's supplied assumptions, the probability of passing given high attendance is approximately **77.78%**. This calculation is separate from the synthetic CSV.

In [ ]:
p_high_given_pass = 0.70
p_high_given_fail = 0.40
p_high = 0.60

p_pass_prior = (p_high - p_high_given_fail) / (p_high_given_pass - p_high_given_fail)
p_pass_given_high = p_high_given_pass * p_pass_prior / p_high

print(f"Inferred P(Pass) = {p_pass_prior:.4f} ({p_pass_prior:.1%})")
print(f"P(Pass | High attendance) = {p_pass_given_high:.4f} ({p_pass_given_high:.2%})")

## 8. Additional exploratory analysis and final summary

Compare pass rates across study-hour and attendance groups, and inspect previous test scores. These are descriptive comparisons, not causal conclusions.

In [ ]:
df["study_group"] = np.where(df["study_hours"] > 10, ">10 hours", "10 hours or less")
df["attendance_group"] = np.where(df["attendance"] > 80, ">80% attendance", "80% or less")

study_pass_rates = df.groupby("study_group", observed=False)["final_exam_pass"].apply(
    lambda s: (s == "Pass").mean()
).rename("Pass rate").reset_index()
attendance_pass_rates = df.groupby("attendance_group", observed=False)["final_exam_pass"].apply(
    lambda s: (s == "Pass").mean()
).rename("Pass rate").reset_index()

display(study_pass_rates)
display(attendance_pass_rates)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=study_pass_rates, x="study_group", y="Pass rate", ax=axes[0])
axes[0].set_title("Pass Rate by Study Hours")
axes[0].set_ylim(0, 1)
sns.barplot(data=attendance_pass_rates, x="attendance_group", y="Pass rate", ax=axes[1])
axes[1].set_title("Pass Rate by Attendance")
axes[1].set_ylim(0, 1)
for ax in axes:
    ax.set_ylabel("Observed pass rate")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

print("Sample size:", N)
print(f"Observed pass rate: {p_marginal_pass:.1%}")
print(f"Pass rate for discussion participants: {p_pass_given_discussion:.1%}")
print(f"Pass rate for non-participants: {(contingency.loc['No','Pass'] / contingency.loc['No'].sum()):.1%}")
print(f"Bayes result from assignment assumptions: {p_pass_given_high:.1%} for high-attendance students")

### Conclusion template — write this in your own words after running the notebook

In this synthetic sample, the observed pass rate is shown above. Compare pass rates by study-hour group, attendance group, previous test performance, and discussion participation before stating which variables show the strongest association with passing. Do not claim causation from these descriptive data. The Bayes result (77.78%) is based on the separate percentages supplied in the assignment, not on the generated CSV.

## 9. Before submission
- Run every cell from top to bottom.
- Check that all tables and charts render.
- Save the notebook with outputs.
- Add your handwritten theory pages as a PDF inside `report/`.
- Add your recorded video URL to `README.md`.
- Commit and push the folder to a public GitHub repository.